# Predictive baselines — classical / feature models

NGBoost, Quantile Regression and Conformal Prediction (CQR), all on the shared
lakehouse plan features. Same data loading / split / targets as the TLSTM and
MLP-head notebooks, so metrics drop straight into the evaluation workbook.

- **NGBoost** and **Quantile Regression** produce a full predictive description
  → evaluated with the Gaussian / quantile metric paths respectively.
- **Conformal** reads point error / CRPS / uncertainty-effectiveness from the
  underlying quantile grid, then overrides Cov@ and MPIW with the calibrated
  intervals.

In [ ]:
import numpy as np
import pandas as pd
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from loader.load import load_aligned_plans_and_runs

from uncertainty_prediction.src import *
from uncertainty_prediction.config import *

from uncertainty_prediction.baselines.predictive.common import (
    build_feature_dataset,
    evaluate_gaussian_predictions,
    evaluate_quantile_predictions,
    override_with_calibrated_intervals,
    print_metric_headers_for_excel,
    print_metrics_for_excel,
    QLEVELS_DEFAULT,
)
from uncertainty_prediction.baselines.predictive.ngboost import NGBoostBaseline
from uncertainty_prediction.baselines.predictive.quantile_regression import QuantileRegression
from uncertainty_prediction.baselines.predictive.conformal import ConformalPrediction

import torch

set_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

In [ ]:
queries_dir = "/mnt/lakehouse-raw-results/tpcds/lakehouse-a/20260222-191819Z/queries"

plans_by_query, runs_by_query, common = load_aligned_plans_and_runs(
    queries_dir=queries_dir,
    run_ids=RUN_IDS,
    collection=COLLECTION_NAME,
    schema=SCHEMA_NAME,
    instance=LAKEHOUSE_INSTANCE_NAME,
    metric=METRIC,
    xcol=XCOL,
    ycol=YCOL,
    parsed_results_root=PARSED_RESULTS_ROOT,
    canon_fn=canon_qid,
    min_runs=1,
    min_points_per_run=2,
    require_cols=(XCOL, YCOL),
)

train_qids, test_qids = split_query_ids(common, seed=SEED, test_frac=TEST_FRAC)
print("n_train:", len(train_qids), "n_test:", len(test_qids))

data = build_feature_dataset(
    plans_by_query=plans_by_query,
    runs_by_query=runs_by_query,
    train_qids=train_qids,
    test_qids=test_qids,
    xcol=XCOL,
    runtime_mode="mean",
)
X_train, X_test = data["X_train"], data["X_test"]
y_train_log, y_test_log = data["y_train_log"], data["y_test_log"]
print("feature_dim:", data["feature_dim"])

## NGBoost
Requires `ngboost` (`pip install ngboost`).

In [ ]:
ng = NGBoostBaseline(n_estimators=500, learning_rate=0.01, seed=42)
ng.fit(X_train, y_train_log)

pred = ng.predict_gaussian(X_test)
ng_metrics = evaluate_gaussian_predictions(pred["mu_log"], pred["sigma_log"], y_test_log)
print_metric_headers_for_excel(ng_metrics)
print_metrics_for_excel(ng_metrics)

## Quantile Regression

In [ ]:
qr = QuantileRegression(in_dim=data["feature_dim"], device=device, seed=42)
qr.fit(X_train, y_train_log, num_epochs=200, lr=1e-3, verbose=True)

Q_test = qr.predict_quantiles(X_test)  # [N, k] log-space quantiles
qr_metrics = evaluate_quantile_predictions(qr.qlevels, Q_test, y_test_log)
print_metric_headers_for_excel(qr_metrics)
print_metrics_for_excel(qr_metrics)

## Conformal Prediction (CQR)

In [ ]:
cp = ConformalPrediction(in_dim=data["feature_dim"], cal_frac=0.3, device=device, seed=42)
cp.fit(X_train, y_train_log, num_epochs=200, lr=1e-3, verbose=True)

# base quantile grid -> point / CRPS / uncertainty-effectiveness
Q_test = cp.predict_quantiles(X_test)
cp_metrics = evaluate_quantile_predictions(cp.qlevels, Q_test, y_test_log)
# override coverage + MPIW with the conformally-calibrated intervals
calibrated = cp.predict_calibrated_intervals(X_test)
cp_metrics = override_with_calibrated_intervals(cp_metrics, calibrated, y_test_log)
print("width corrections (log space):", cp._corrections)
print_metric_headers_for_excel(cp_metrics)
print_metrics_for_excel(cp_metrics)

## Side-by-side summary

In [ ]:
summary = pd.DataFrame(
    {
        "NGBoost": ng_metrics,
        "Quantile Regression": qr_metrics,
        "Conformal Prediction": cp_metrics,
    }
).T
cols = ["mae", "rmse", "median_q_error", "crps", "cov@50", "cov@90", "cov@99", "mpiw", "unc_spearman", "unc_pearson"]
summary[cols]